In [ ]:
# Carregando bibliotecas e base de dados
from ucimlrepo import fetch_ucirepo

import seaborn as sns
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Perceptron
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.metrics import f1_score
from sklearn.metrics import recall_score
from sklearn.metrics import precision_score
from sklearn.metrics import precision_recall_curve

In [ ]:
# fetch dataset 
diabetes = fetch_ucirepo(id=296)

In [ ]:
# data (Pandas dataframes) 
X = diabetes.data.features

# Criando a variável indicadora de ausência de peso e removendo a coluna 'weight' por conter mais de 96% de valores nulos
X['weight_missing'] = X['weight'].isnull().astype(int)

# target (Pandas series)
y = diabetes.data.targets['readmitted']

# Transformando a variável alvo em binária (1 para '<30' e 0 para 'NO' e '>30')
y = y.apply(lambda x: 1 if x == '<30' else 0)

# Dividindo os dados em conjuntos de treino e teste
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

# metadata 
x_train.info()

In [ ]:
# Explorando as primerias linhas do dataset
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
print(x_train.head(200))

# Verificando o tipo do dataset como dataframe do Pandas
type(X)

In [ ]:
# Dando uma geral no dataset
x_train.info()

In [ ]:
display(x_train.isnull().sum())

# Percentual de nulos por coluna
display(x_train.isna().sum() / X.shape[0])

Coluna Weight tem quase 68% de dados nulos, o que implica que remover todas essas linhas inviabilizaria o treinamento do modelo. Como o peso é uma variável importante no controle da glicose, seria imprudente colocar um valor mediano ou outras formas estatísticas.
Imputar média/mediana vai basicamente criar uma coluna quase constante. O modelo pode aprender ruído. Pode até prejudicar a performance.
Decidi remover a coluna devido ao número alto de valores nulos, evitando ruído e nao distorcendo a distribuíçao.

In [ ]:
cat_unknown = ['medical_specialty', 'payer_code']
cat_not_measured = ['A1Cresult', 'max_glu_serum']

In [ ]:
# Removendo a coluna 'weight' por conter mais de 96% de valores nulos e criando o indicador de ausência de peso
x_train['weight_missing'] = x_train['weight'].isna().astype(int)
x_train = x_train.drop(columns=['weight'], axis=1)

In [ ]:
# Preenchendo os valores nulos das colunas 'A1Cresult' e 'max_glu_serum' com a categoria 'NotMeasured'
x_train['A1Cresult'] = x_train['A1Cresult'].fillna('NotMeasured')
x_train['max_glu_serum'] = x_train['max_glu_serum'].fillna('NotMeasured')

# Preenchendo os valores nulos das colunas 'medical_specialty' e 'payer_code' com a categoria 'Unknown'
x_train['medical_specialty'] = x_train['medical_specialty'].fillna('Unknown')
x_train['payer_code'] = x_train['payer_code'].fillna('Unknown')

In [ ]:
# Criando listas de colunas categóricas para cada tipo de tratamento de valores nulos
null_cats = ['diag_1', 'diag_2', 'diag_3', 'race']

In [ ]:
# Removendo as linhas com valores nulos nas colunas de diagnóstico e race
x_train = x_train.dropna(subset=null_cats)
y_train = y_train.loc[x_train.index]  # MUITO IMPORTANTE alinhar o target

In [ ]:
# Confirmando que não há mais valores nulos
display(x_train.isna().sum() / x_train.shape[0])

In [ ]:
#total de observações após remoção de nulos
print(f'Total de observações após remoção de nulos: {x_train.shape[0]}')

In [ ]:
numeric_cols = x_train.select_dtypes(include=['int64']).columns
categorical_cols = x_train.select_dtypes(include=['object']).columns

In [ ]:
# Criando pipelines separados para as colunas categóricas com tratamento de valores nulos
cat_unknown_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Criando pipelines separados para as colunas categóricas com tratamento de valores nulos
cat_not_measured_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='NotMeasured')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

In [ ]:
# Utilizando o ColumnTransformer para aplicar as transformações necessárias em cada tipo de variável
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_cols),
        ('cat_unknown', cat_unknown_pipeline, cat_unknown),
        ('cat_not_measured', cat_not_measured_pipeline, cat_not_measured),
        # handle_unknown='ignore' é necessário para evitar erros caso haja categorias no conjunto de teste que não estejam presentes no conjunto de treino
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ]
)

In [ ]:
# Criando o pipeline com o pré-processamento e o modelo de classificação com Perceptron
model = Pipeline(steps=[
    ('preprocessing', preprocessor),
    ('classifier', Perceptron(class_weight='balanced', random_state=42, max_iter=1000))
])

In [ ]:
# Analisando a distribuição da variável alvo
display(y_train.value_counts())

# Analisando as proporções da variável alvo
display(y_train.value_counts(normalize=True))

In [ ]:
# Verificando o formato dos dados
display(x_train.shape)
# Verificando o formato da variável alvo
display(y_train.shape)
display(y_train.head())

# Treinando com Perceptron
display(model.fit(x_train, y_train))

# Fazendo previsões
y_hat_train = model.predict(x_train)
y_hat_test = model.predict(x_test)

print("Acurácia no conjunto de treino: ", accuracy_score(y_train, y_hat_train))
print("Acurácia no conjunto de teste: ", accuracy_score(y_test, y_hat_test))

print("-"*100)

y_pred = model.predict(x_test)

print(y_test.dtype)
print(y_pred.dtype)

print(y_test.unique())
print(set(y_pred))

print("Matriz de confusão (Perceptron):")
print(confusion_matrix(y_test, y_pred))
print("-"*100)
print("Classification Report (Perceptron):")
print(classification_report(y_test, y_pred))

### Análise do modelo

* O modelo está basicamente classificando quase todo mundo como classe 1

* Recall da classe 1 = 0.96 (quase pega todos)
* Precision = 0.12 (muitos falsos positivos)
* Acurácia geral em 17%

In [ ]:
# Criando o pipeline com o pré-processamento e o modelo de classificação com Regressão Logística
model = Pipeline(steps=[
    ('preprocessing', preprocessor),
    ('classifier', LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000))
])
# Treinando o modelo de regressão logística
model.fit(x_train, y_train)

In [ ]:
# Fazendo previsões

y_hat_train = model.predict(x_train)
y_hat_test = model.predict(x_test)

In [ ]:
print(y_hat_train[:5])
print(y_train.head())

In [ ]:
print("Acurácia no conjunto de treino: ", accuracy_score(y_train, y_hat_train))
print("Acurácia no conjunto de teste: ", accuracy_score(y_test, y_hat_test))

In [ ]:
y_pred = model.predict(x_test)

print("-"*100)
print("Matriz de confusão (Regressão Logística):")
print(confusion_matrix(y_test, y_pred))
print("-"*100)
print("Classification Report (Regressão Logística):")
print(classification_report(y_test, y_pred))

# Análise técnica dos modelos
* Perceptron
    * Modelo linear simples
    * Recall alto para classe 1 (96%)
    * Muitos falsos positivos
    * Acurácia baixa (17%)
    * Precisao muito baixa para classe 1 (12%)
    
* Regressao Logística
    * Modelo linear probabilístico, mais estável
    * class_weight=balanced funciona muito melhor
    * Recall da classe 1 em 50%
    * Permite ajustar limiar de decisao
    * Muito mais apropriado para problemas de desbalanceamento

Embora o Perceptron apresente Recall de 96%, a acurácia ficou muito baixa (17%).
O número de falso-positivos aprensentou um número muito alto (25133)

A Regressão Logística, ao utilizar class_weight='balanced', apresenta um resultado melhor,
apesar do recall da classe 1 ficar em 50%, reduzindo substancialmente o valor de falsos-positivo (8459)

In [ ]:
# Pairplot para analisar as relações entre as variáveis numéricas
df_plot = X.copy()
df_plot['readmitted'] = y.astype(str)
sns.pairplot(df_plot.sample(1500),
             vars=numeric_cols,
             diag_kind='kde',
             hue='readmitted',
             palette='Set2')

In [ ]:
# Testando com KNN

knn_model = Pipeline(steps=[
    ('preprocessing', preprocessor),
    ('classifier', KNeighborsClassifier(n_neighbors=3))
])

knn_model.fit(x_train, y_train)

In [ ]:
y_hat_train = knn_model.predict(x_train)
y_hat_test = knn_model.predict(x_test)

In [ ]:
print("Acurácia no conjunto de treino: ", accuracy_score(y_train, y_hat_train))
print("Acurácia no conjunto de teste: ", accuracy_score(y_test, y_hat_test))

In [ ]:
print("-"*100)
print("Matriz de confusão (KNN):")
print(confusion_matrix(y_test, y_hat_test))
print("-"*100)
print("Classification Report (KNN):")
print(classification_report(y_test, y_hat_test))

# _Comparaçao dos Modelos:_

* KNN
    * Tem acurácia alta
    * Praticamente ignora a classe minoritária
    * Recall muito baixo (Detecta apenas 9% dos pacientes de risco)
    * Muitas features categóricas explodem dimensionalidade
    * Falso Negativo muito alto, acerta apenas 290
---
    Está favorecendo a classe majoritária
---
* Perceptron (balanceada)
    * Acurária baixa
    * Recall muito alto baixo
    * Muitos falsos-positivos
---
    Recall altíssimo, mas gera falsos-positivos demais.
---
* Regressao Logística (balanceada)
    * Acurácia aceitável
    * Recall razoável
    * Melhor F1-score da classe 1

    ### Melhor até agora
---
* Accuracy pode ser enganosa em datasets desbalanceados 
* Modelos não balanceados favorecem a classe majoritária 
* Ajuste de class_weight muda completamente o comportamento do modelo
---

In [ ]:
# Pegando as probabilidades preditas para a classe 1 (readmitidos em menos de 30 dias)
y_prob = model.predict_proba(x_test)[:, 1]

In [ ]:
# Testando diferentes thresholds para a classificação
# O threshold padrão é 0.5, ou seja, se a probabilidade predita for maior ou igual a 0.5, 
# o modelo classifica como 1 (readmitido), caso contrário, classifica como 0 (não readmitido).
# O melhor threshold depende do trade-off entre precisão e recall que queremos alcançar, 
# e pode ser ajustado de acordo com as necessidades do problema em questão.
# O threshold de 0.60 é o que apresenta melhor equilíbrio entre precisão e recall para a classe 1

# Em contexto hospitalar normalmente o recall é mais importante que a precisão, pois perder 
# um paciente de risco pode gerar custo e complicação clínica.
# Apesar do threshold de 0.50 apresentar F1 score ligeiramente mais alto, a precisao cai para 0.163
# O threshold de 0.60 é o que apresenta melhor equilíbrio entre precisão e recall e F1 score para a classe 1

# O threshold de 0.45 apresenta um recall de 0.648, mas a precisão cai para 0.152. Buscando um balanceado com F1 score

thresholds = np.arange(0.05, 0.9, 0.05)

results = []

for t in thresholds:
    y_pred_adj = (y_prob >= t).astype(int)
    
    recall = recall_score(y_test, y_pred_adj)
    precision = precision_score(y_test, y_pred_adj)
    f1 = f1_score(y_test, y_pred_adj)
    
    results.append((t, recall, precision, f1))

for r in results:
    print(f"Threshold: {r[0]:.2f} | Recall: {r[1]:.3f} | Precision: {r[2]:.3f} | F1: {r[3]:.3f}")

In [ ]:
# Filtrando os resultados para identificar os thresholds que apresentam recall maior ou igual a 0.60
filtered = [r for r in results if r[1] >= 0.60 and r[3] >= 0.24]
filtered

In [ ]:
best_threshold = 0.50

y_final = (y_prob >= best_threshold).astype(int)

print(classification_report(y_test, y_final))

Recall = 0.62 (bem alto)

* F1 = 0.25 (muito próximo dos melhores scores)

* Captura mais pacientes de risco

* A precisão é baixa (0.15), mas isso é esperado com base desbalanceada.

No geral, o threshold de 50% tem os melhores classificadores gerais.

In [ ]:
precision, recall, thresholds = precision_recall_curve(y_test, y_prob)

f1_scores = 2 * (precision * recall) / (precision + recall)
best_index = np.argmax(f1_scores)

best_threshold = thresholds[best_index]
print("Best threshold:", best_threshold)

display(thresholds)

In [ ]:
# Aplicando o melhor threshold encontrado
y_final = (y_prob >= 0.45).astype(int)

print(classification_report(y_test, y_final))

In [ ]:
threshold = 0.50

# Probabilidades da classe 1
y_prob = model.predict_proba(x_test)[:, 1]

# Classificação com novo threshold
y_pred_new = (y_prob >= threshold).astype(int)

# Criar DataFrame
df_plot = pd.DataFrame({
    "Probabilidade": y_prob,
    "Predição (threshold=0.45)": y_pred_new
})

# Boxplot
plt.figure(figsize=(8,5))
sns.boxplot(x="Predição (threshold=0.45)", y="Probabilidade", data=df_plot)
plt.axhline(threshold, color='red', linestyle='--', label='Threshold 0.45')
plt.legend()
plt.title("Distribuição das Probabilidades por Classe Predita")
plt.show()

* Classe 0 predita → probabilidades concentradas abaixo de 0.5

* Classe 1 predita → probabilidades acima de 0.5

* A linha vermelha está exatamente no ponto de decisão

Isso confirma que o threshold está sendo aplicado corretamente.

* A mediana da classe 1 está por volta de 0.55–0.60

* A mediana da classe 0 está ~0.35

* Há separação, mas não é enorme, por isso o recall e precision não são muito altos.

In [ ]:
df_plot = pd.DataFrame({
    "Probabilidade": y_prob,
    "Classe Real": y_test
})

plt.figure(figsize=(8,5))
sns.boxplot(x="Classe Real", y="Probabilidade", data=df_plot)
plt.axhline(threshold, color='red', linestyle=':')
plt.title("Probabilidade por Classe Real")
plt.show()

In [ ]:
# As distribuições das probabilidades das duas classes estão muito sobrepostas.
# O modelo não separa bem as classes.
# A mediana da classe 1 está praticamente em 0.5.
# Isso indica que o modelo está em dúvida sobre a maioria dos casos positivos.

In [ ]:
plt.figure(figsize=(8,5))

sns.kdeplot(df_plot[df_plot["Classe Real"]==0]["Probabilidade"], label="Classe 0")
sns.kdeplot(df_plot[df_plot["Classe Real"]==1]["Probabilidade"], label="Classe 1")

plt.axvline(threshold, color='red', linestyle='--')
plt.title("Distribuição de Probabilidade por Classe Real")
plt.legend()
plt.show()

In [ ]:
pipe_dt = Pipeline([
    ('preprocessing', preprocessor),
    ('classifier', DecisionTreeClassifier(class_weight='balanced', random_state=42))
])

params_grid = {
    'classifier__criterion': ['gini', 'entropy', 'log_loss'],
    # Esse range eu escolhi porque testei com alguns valores fixos, que continha o número 5. Refiz om range para confirmar.
    'classifier__max_depth': range(1, 7),
    # Esses parametros nao influenciaram muito o desempenho do modelo, mas mantive para testar
    'classifier__splitter': ['best', 'random'],
    'classifier__class_weight': ['balanced', None]
}

In [ ]:
# Configurando um amostrador estratificado para validação cruzada

splitter = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

In [ ]:
# Configurando o experimentador de hiperparâmetros com GridSearchCV

grid_search = GridSearchCV(
    estimator=pipe_dt, 
    param_grid=params_grid, 
    cv=splitter, 
    scoring='f1', 
    n_jobs=-1, 
    refit=True,
    error_score=0
)

In [ ]:
# Realizando o experimento

grid_search.fit(x_train, y_train)

In [ ]:
# Analisando a melhor combinação de hiperparâmetros encontrada

grid_search.best_params_

In [ ]:
# Qual o melhor desempenho alcançado com a melhor combinação de hiperparâmetros?

grid_search.best_score_

In [ ]:
# Analisando a precisao do fold do melhor modelo encontrado

precision_cv = []
for i in range(10):
    precision_fold = grid_search.cv_results_['split{}_test_score'.format(i)][grid_search.best_index_]
    precision_cv.append(precision_fold)

print("Precisão por fold:", precision_cv)
print("Desvio padrão da precisão nos folds:", np.std(precision_cv))
print("Coefficiente de variação da precisão nos folds:", np.std(precision_cv) / np.mean(precision_cv))
print("95% do intervalo de confiança:", (1.96 * np.std(precision_cv) / np.sqrt(len(precision_cv))))

In [ ]:
yhat_train = grid_search.predict(x_train)
yhat_test = grid_search.predict(x_test)

# Analisando a acurácia do modelo de árvore de decisão com os melhores hiperparâmetros encontrados
print("Acurácia no conjunto de treino: ", accuracy_score(y_train, yhat_train))
print("Acurácia no conjunto de teste: ", accuracy_score(y_test, yhat_test))
print("="*100)
print("Classification Report (Árvore de Decisão):\n", classification_report(y_test, yhat_test))

In [ ]:
# Trouxe acurácia semelhante à Regressão Logística, mas com um comportamento bem diferente nas classes
# Nao está overfitado - Treino e teste bem semelhantes
# Apresentou melhor desempenho na classe 1 com recall de 60% e precisão de 25% (melhor que os outros modelos testados)
# F1 também é o melhor
# Acurácia baixa causando muitos falsos positivos, mas em cenários de saúde onde o custo de um falso negativo é muito alto,
# esse comportamento pode ser aceitável
# Apesar de baixa acurácia global, o modelo se mostrou eficaz para identificar pacientes com risco de readmissao
# ainda que com alta taxa de falsos positivos.

In [ ]:
pipe_dt_bigger = Pipeline([
    ('preprocessing', preprocessor),
    ('classifier', DecisionTreeClassifier(class_weight='balanced', random_state=42))
])

params_grid2 = {
    'classifier__criterion': ['gini', 'entropy', 'log_loss'],
    # Esse range eu escolhi porque testei com alguns valores fixos, que continha o número 5. Refiz om range para confirmar.
    'classifier__max_depth': range(1, 7),
    # Esses parametros nao influenciaram muito o desempenho do modelo, mas mantive para testar
    'classifier__splitter': ['best', 'random'],
    'classifier__class_weight': ['balanced', None]
}

In [ ]:
# Configurando o experimentador de hiperparâmetros com GridSearchCV

grid_search2 = RandomizedSearchCV(
    estimator=pipe_dt_bigger, 
    param_distributions=params_grid2, 
    cv=splitter, 
    scoring='f1', 
    n_jobs=-1, 
    refit=True,
    error_score=0,
    # O número de iteracoes maximo é de 72. Deixei assim mesmo.
    n_iter=200,
    random_state=42
)

In [ ]:
grid_search2.fit(x_train, y_train)

In [ ]:
grid_search2.best_params_

In [ ]:
grid_search2.best_score_

In [ ]:
# Analisando a precisao do fold do melhor modelo encontrado

precision_cv2 = []
for i in range(10):
    precision_fold = grid_search2.cv_results_['split{}_test_score'.format(i)][grid_search2.best_index_]
    precision_cv2.append(precision_fold)

print("Precisão por fold:", precision_cv2)
print("Desvio padrão da precisão nos folds:", np.std(precision_cv2))
print("Coefficiente de variação da precisão nos folds:", np.std(precision_cv2) / np.mean(precision_cv2))
print("95% do intervalo de confiança:", (1.96 * np.std(precision_cv2) / np.sqrt(len(precision_cv))))

In [ ]:
yhat_train = grid_search2.predict(x_train)
yhat_test = grid_search2.predict(x_test)

# Analisando a acurácia do modelo de árvore de decisão com os melhores hiperparâmetros encontrados
print("Acurácia no conjunto de treino: ", accuracy_score(y_train, yhat_train))
print("Acurácia no conjunto de teste: ", accuracy_score(y_test, yhat_test))
print("="*100)
print("Classification Report (Árvore de Decisão):\n", classification_report(y_test, yhat_test))